In [1]:
import os

# import sys; sys.path.append("/lustre1/chengqiyi_pkuhpc/zhaohn/0.apps")
import time

# merge bmat files

In [2]:
# !python /lustre1/chengqiyi_pkuhpc/zhaohn/0.apps/BioinformaticAnalysisTools/TargetSeq_Tools/get_target-seq_info.py -h

In [3]:
# # 执行下面的操作前完成bmat的搜集和重命名
# os.makedirs("../bmat", exist_ok=True)

# os.chdir("..")
# # !find . -type f -name "*.bmat" | awk '{printf "cp "$0" "; sub(/./, "bmat"); gsub(/\//, "_"); sub(/_/, "/"); print $0 }'

# !find . -type f -name "*.bmat" | awk '{printf "cp "$0" "; sub(/./, "bmat"); gsub(/\//, "_"); sub(/_/, "/"); print $0 }' | sh
# os.chdir("snakepipes_target-seq")

In [4]:
# 用sublime等完成重命名
# eg:
# TargetSeq-TREATMENT_ABE_REP_2_CUTOFF_3_REGIONID_HK4-Rank-33.bmat.gz


# gzip ../bmat/*

In [5]:
today = time.strftime("%Y-%m-%d", time.localtime())
today

'2022-08-03'

In [6]:
# !python /lustre1/chengqiyi_pkuhpc/zhaohn/0.apps/BioinformaticAnalysisTools/TargetSeq_Tools/get_target-seq_info.py \
#     --bmat_folder ../bmat \
#     --out '{today}.bmats.csv'

# analysis

## imports

In [7]:
# !pip install numpy pandas matplotlib seaborn openpyxl lets-plot statannotations

In [38]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statannotations.Annotator import Annotator
from pandarallel import pandarallel

sns.set()
pd.set_option("max_colwidth", 250)  # column最大宽度
pd.set_option("display.width", 250)  # dataframe宽度
pd.set_option("display.max_columns", None)  # column最大显示数
pd.set_option("display.max_rows", 50)  # row最大显示数
pandarallel.initialize(nb_workers=12)  # 多线程设置，默认使用全部核心 nb_workers=24

INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


## preview

In [30]:
df = pd.read_csv(f"{today}.bmats.csv")

# drop names
df.drop("bmat_name", axis=1, inplace=True)

print(df.head(), "\n"*2, "*" * 100)
print(df.info(), "\n"*2, "*" * 100)
print(df.isnull().sum(), "\n", "*" * 100)

      treatment   rep  cutoff     region_id ref_base  relative_pos  total_count mut_base  mut_count
0  ND5.1-sTALED  rep1       3  share-new-11        C             1       250101        A          0
1  ND5.1-sTALED  rep1       3  share-new-11        A             2       250097        A     250097
2  ND5.1-sTALED  rep1       3  share-new-11        T             3       250101        A          0
3  ND5.1-sTALED  rep1       3  share-new-11        C             4       250100        A          0
4  ND5.1-sTALED  rep1       3  share-new-11        C             5       250100        A          0 

 ****************************************************************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107768 entries, 0 to 107767
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   treatment     107768 non-null  object
 1   rep           107768 non-null  object
 2   cutoff   

In [31]:
# 查看treatment
print(df.treatment.value_counts())

untreat-Q5U     44384
ND6-Det-Q5U     30096
ND5.1-sTALED    11096
ND1-sTALED      11096
ND4-sTALED      11096
Name: treatment, dtype: int64


## barplot: background base mutation of different treatment

In [33]:
df['direction'] = df.ref_base + "2" + df.mut_base
df['mut_ratio'] = df.mut_count / df.total_count * 100
df = df.query("""
    ref_base != mut_base & \
    cutoff == 3 & \
    mut_ratio <= 20
""")
df.head()

,treatment,rep,cutoff,region_id,ref_base,relative_pos,total_count,mut_base,mut_count,direction,mut_ratio
0,ND5.1-sTALED,rep1,3,share-new-11,C,1,250101,A,0,C2A,0.0
2,ND5.1-sTALED,rep1,3,share-new-11,T,3,250101,A,0,T2A,0.0
3,ND5.1-sTALED,rep1,3,share-new-11,C,4,250100,A,0,C2A,0.0
4,ND5.1-sTALED,rep1,3,share-new-11,C,5,250100,A,0,C2A,0.0
5,ND5.1-sTALED,rep1,3,share-new-11,C,6,250102,A,0,C2A,0.0


In [34]:
# barplot

# 变量
x = "direction"
y = "mut_ratio"
hue = "treatment"
# hue_order = None
hue_order = ["ND1-sTALED", "ND4-sTALED", "ND5.1-sTALED", "ND6-Det-Q5U", "untreat-Q5U"]
width = 0.4

palette = ['#B42318', '#E4453A', '#EE8981', '#94BFA7', '#EDD4D4']
# palette = "Set2"

ax = sns.barplot(
    data=df, x=x, y=y, hue=hue, hue_order=hue_order, palette=palette,
    estimator=np.mean, # callable that maps vector -> scalar, optional  # 在每个分类bin内进行估计的统计函数
    ci=None,  # 用于绘制估计值的置信区间的大小。如果是“sd”，则跳过bootstrapping并绘制观测值的标准差。如果为None，则不会执行引导，也不会绘制错误条。
    edgecolor="k",  # 描边颜色
    linewidth=0.2,  # 描边宽度
)
ax.axhline(y=0.025, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], c='blue', linestyle='--', label='X-TEN max')
ax.axhline(y=0.050, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], c='red', linestyle='--', label='MGI-2000 max')



# 表头和标注
ax.set_title("Target deep sequencing mutation direction (All region, rm SNP>20%)", fontsize=20)
ax.set_xlabel("Mutation direction")
ax.set_ylabel("Mutation ratio (%)")

# 添加检验

# 检验配对
# pairs = [((p, 'MGI2000'), (p, 'X-TEN')) for p in df['direction'].unique()]
# annot = Annotator(ax, pairs, plot='barplot', data=df, x=x, y=y, hue=hue, hue_order=hue_order, seed=2021)
# 定义为t检验
# annot.configure(test='t-test_ind', verbose=2)
# 进行检验
# annot.apply_test()
# 添加标注
# annot.annotate()



# 设置legend
ax.legend(loc='upper left', bbox_to_anchor=(1.03, 1), title=hue)


# 绘图风格
# plt.rcParams['font.family'] = ['Times']
# plt.rcParams["axes.labelsize"] = 18
# plt.rcParams["axes.titlesize"] = 308
plt.rcParams['figure.figsize'] = [12, 7]
# 储存
plt.savefig(f'{today}_target-seq_barplot_mutation-direction.pdf', dpi=300, bbox_inches='tight')

plt.close()

## barplot: strong signal

In [47]:
# 在此之前进行了过滤SNP，要注意 mut_ratio <= 20 cutoff == 3
print(df.head())
df.treatment.unique()

      treatment   rep  cutoff     region_id ref_base  relative_pos  total_count mut_base  mut_count direction  mut_ratio
0  ND5.1-sTALED  rep1       3  share-new-11        C             1       250101        A          0       C2A        0.0
2  ND5.1-sTALED  rep1       3  share-new-11        T             3       250101        A          0       T2A        0.0
3  ND5.1-sTALED  rep1       3  share-new-11        C             4       250100        A          0       C2A        0.0
4  ND5.1-sTALED  rep1       3  share-new-11        C             5       250100        A          0       C2A        0.0
5  ND5.1-sTALED  rep1       3  share-new-11        C             6       250102        A          0       C2A        0.0


array(['ND5.1-sTALED', 'ND1-sTALED', 'ND6-Det-Q5U', 'untreat-Q5U',
       'ND4-sTALED'], dtype=object)

In [48]:
def mut_ratio_filter(x):
    if x.treatment == 'untreat-Q5U':
        return True
    else:
        if x.mut_ratio >= 0.01:  # 0.01%
            return True
        else:
            return False

df_signal = df[df.parallel_apply(mut_ratio_filter, axis=1)].copy()
df_signal.head()

,treatment,rep,cutoff,region_id,ref_base,relative_pos,total_count,mut_base,mut_count,direction,mut_ratio
58,ND5.1-sTALED,rep1,3,share-new-11,C,59,250098,A,32,C2A,0.012795
59,ND5.1-sTALED,rep1,3,share-new-11,C,60,250101,A,35,C2A,0.013994
127,ND5.1-sTALED,rep1,3,share-new-11,G,128,250099,A,44,G2A,0.017593
172,ND5.1-sTALED,rep1,3,share-new-11,C,173,250099,A,34,C2A,0.013595
253,ND5.1-sTALED,rep1,3,share-new-11,C,254,250100,A,123,C2A,0.049180


In [49]:
# filter cutoff3

# merge rep, direction
# mut_ratio是百分数


df_signal = df_signal.groupby(['treatment', 'region_id', 'direction']) \
    ['mut_ratio'] \
    .agg(['max', 'mean']) \
    .reset_index()
df_signal

,treatment,region_id,direction,max,mean
0,ND1-sTALED,ND516-share-10,A2C,0.086011,0.086011
1,ND1-sTALED,ND516-share-10,C2A,0.030357,0.017347
2,ND1-sTALED,ND516-share-10,C2T,0.049874,0.032676
3,ND1-sTALED,ND516-share-10,G2A,0.033264,0.018797
4,ND1-sTALED,ND516-share-10,G2T,0.017347,0.016865
...,...,...,...,...,...
498,untreat-Q5U,share-new-9,G2C,0.003997,0.000234
499,untreat-Q5U,share-new-9,G2T,0.025314,0.001043
500,untreat-Q5U,share-new-9,T2A,0.002665,0.000160
501,untreat-Q5U,share-new-9,T2C,0.018652,0.000584


In [51]:
from lets_plot import *
LetsPlot.setup_html()

In [145]:
# g = (ggplot() + 
#      geom_bar(data=df_signal,
#               mapping=aes(x='direction', y='mean', fill='direction'),
#               stat='identity',
#               position='dodge') + 
#      facet_grid(x = 'region_id', y='treatment', scales='free_y') + 
#      # scale_x_log10() + 
#      scale_y_continuous(limits=[0, 3], breaks=np.arange(0,3,0.2), format='.2f') + 
#      theme_classic() + 
#      ylab("Mutation ratio (%)") +
#      ggtitle("Targeted deep sequencing ratio (mean)") +
#      theme(axis_text=element_text(color='#bdbdbd'))
# )
# g

# # ggsave(g, f'{today}_target-seq_barplot_mutation-direction_details.html', path='.', iframe=True)

In [169]:
g = (ggplot() + 
     geom_bar(data=df_signal,
              mapping=aes(x='direction', y='max', fill='direction'),
              stat='identity',
              position='stack') + 
     facet_grid(x = 'region_id', y='treatment', scales='free_y') + 
     # scale_x_log10() + 
     # scale_y_continuous(limits=[0, 3], breaks=np.arange(0,3,0.2), format='.2f') + 
     theme_classic() + 
     ylab("Mutation ratio (%)") +
     ggtitle("Targeted deep sequencing ratio (highest-base on the amplicon)") +
     theme(axis_text=element_text(color='#bdbdbd'))
)
g

In [170]:
ggsave(g, f'{today}_target-seq_barplot_mutation-direction_details.html', path='.', iframe=False)

'/lustre1/chengqiyi_pkuhpc/zhaohn/3.project/2022_ABE_topic/TargetSeq/2022-07-31_TargetSeq_batch3_sTALED/snakepipes_target-seq/2022-08-03_target-seq_barplot_mutation-direction_details.html'

## barplot: use aim index

In [153]:
# # %%load base index
# # 每个off-target region中指定base的TargetSeq ratio
# df_idx = pd.read_excel(
#     "2022-05-24_ABE_only_one_mut_index_info.xlsx", sheet_name="Sheet1"
# )
# # print(df_idx)
# print(df_idx.info())
# df_idx = df_idx[["region_id", "relative_pos"]].copy()
# print(df_idx)
# df_idx.groupby("region_id").relative_pos.count().sum()

In [154]:
# # %% merge all-target table and index table
# df_one_base = df_idx.merge(df, how="left", on=["region_id", "relative_pos"])
# print(df_one_base)
# df_one_base.groupby(["region_id", "treatment"]).count()

In [155]:
# # check df_one_base
# # 44个点,没错
# print(df_one_base.region_id.value_counts().index.__len__())

In [156]:
# # %% calculating
# # %%% 计算mut_ratio * 100
# df_one_base["mut_ratio"] = df_one_base.mut_count / df_one_base.total_count * 100
# df_one_base

# df_one_base.groupby(["region_id", "treatment"]).count()

In [157]:
# # %%% filter A2G or T2C
# df_one_base_filter_base = df_one_base[
#     ((df_one_base.ref_base == "A") & (df_one_base.mut_base == "G"))
#     | (df_one_base.ref_base == "T") & (df_one_base.mut_base == "C")
# ].copy()
# df_one_base_filter_base

In [158]:
# # %%% filter 用cutoff3(off) cutoff0(on)
# def filter_cutoff(x):
#     if (x["cutoff"] == 3) and ("on-target" not in x["region_id"]):
#         return True
#     elif (x["cutoff"] == 0) and ("on-target" in x["region_id"]):
#         return True
#     else:
#         return False


# df_one_base_filter_base = df_one_base_filter_base[
#     df_one_base_filter_base.apply(filter_cutoff, axis=1)
# ].copy()

# df_use = df_one_base_filter_base.copy()
# df_use

In [159]:
# # check df_one_base
# # 44个点,没错, 因为on-target不止count了一次
# print(df_use.region_id.value_counts().index.__len__())
# df_use.head()

In [160]:
# df_use["mut_info"] = df_use.ref_base + df_use.relative_pos.map(str)
# df_use

In [161]:
# # %%% 计算 merge rep后的 mean ratio
# df_use1 = (
#     df_use.groupby(["region_id", "relative_pos", "treatment"])
#     .mut_ratio.mean()
#     .reset_index()
# )
# df_use1.rename(columns={"mut_ratio": "mut_ratio_mean"}, inplace=True)
# df_use1

In [162]:
# df_use2 = pd.merge(
#     left=df_use[
#         ["region_id", "relative_pos", "treatment", "rep", "mut_info", "mut_ratio"]
#     ],
#     right=df_use1,
# )

# df_plot = df_use2.copy()
# df_plot

In [163]:
# # %% plot
# # 配色 https://blog.csdn.net/black000shirt/article/details/113724245


# def off_barplot(df):
#     fig = (
#         p9.ggplot()
#         + p9.geom_bar(
#             data=df,
#             mapping=p9.aes(x="region_id", y="mut_ratio_mean", fill="treatment"),
#             stat=p9.stats.stat_identity,
#             position=p9.positions.position_dodge(
#                 width=0.9, preserve="total"  # Bar width
#             ),
#             color="black",
#             size=0.1,
#             raster=False,
#         )
#         + p9.geom_point(
#             data=df,
#             mapping=p9.aes(
#                 x="region_id",
#                 y="mut_ratio",
#                 # fill="treatment",
#                 group="treatment",
#             ),
#             # color="black",
#             position=p9.positions.position_dodge(
#                 width=0.9, preserve="total"  # Bar width
#             ),
#             size=0.2,
#         )
#         + p9.scale_x_discrete(name="Off-target sites")
#         + p9.scale_y_continuous(
#             name="Editing ratio by Targeted deep sequencing (%)",
#             # breaks=np.arange(0, df.mut_ratio.max(), round(df.mut_ratio.max() / 10, 1)),
#             labels=lambda L: ["%.1f" % v for v in L],
#         )
#         + p9.coord_flip()
#         + p9.theme_classic()
#         # + p9.scale_fill_brewer(type="qualitative", palette="Set2")  # 画share old的时候注释掉这一行不然颜色不够用
#         + p9.ggtitle("Targeted deep sequencing ratio")
#     )
#     return fig


# def off_barplot_log10(df):
#     # fix log10
#     df["mut_ratio"] = np.log10(df.mut_ratio) + 5
#     df["mut_ratio"] = df.mut_ratio.map(lambda x: x if x > 0 else 0)
#     df["mut_ratio_mean"] = np.log10(df.mut_ratio_mean) + 5
#     df["mut_ratio_mean"] = df.mut_ratio_mean.map(lambda x: x if x > 0 else 0)
#     # plot
#     fig = (
#         p9.ggplot()
#         + p9.geom_bar(
#             data=df,
#             mapping=p9.aes(x="region_id", y="mut_ratio_mean", fill="treatment"),
#             stat=p9.stats.stat_identity,
#             position=p9.positions.position_dodge(
#                 width=0.9, preserve="total"  # Bar width
#             ),
#             color="black",
#             size=0.1,
#             raster=False,
#         )
#         + p9.geom_point(
#             data=df,
#             mapping=p9.aes(
#                 x="region_id",
#                 y="mut_ratio",
#                 # fill="treatment",
#                 group="treatment",
#             ),
#             # color="black",
#             position=p9.positions.position_dodge(
#                 width=0.9, preserve="total"  # Bar width
#             ),
#             size=0.2,
#         )
#         + p9.scale_x_discrete(name="Off-target sites")
#         + p9.scale_y_continuous(
#             # fix log10
#             limits=np.log10([0.00001, 100]) + 5,
#             breaks=np.log10([0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100]) + 5,
#             labels=[0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
#             name="log10(Editing ratio by Targeted deep sequencing (%))",
#         )
#         + p9.coord_flip()
#         + p9.theme_classic()
#         # + p9.scale_fill_brewer(type="qualitative", palette="Set2")  # 画share old的时候注释掉这一行不然颜色不够用
#         + p9.ggtitle("Targeted deep sequencing ratio")
#     )
#     return fig


# def off_jitterplot(df):
#     fig = (
#         p9.ggplot(
#             data=df,
#             mapping=p9.aes(x="treatment", y="mut_ratio", fill="treatment"),
#         )
#         + p9.geom_jitter(
#             **{
#                 "stat": "identity",
#                 "na_rm": False,
#                 "width": 0.1,
#                 "height": 0,
#                 "random_state": None,
#             }
#         )
#         + p9.geom_boxplot(alpha=0.2)
#         + p9.scale_y_continuous(breaks=np.arange(0, df.mut_ratio.max(), 0.5))
#         + p9.theme_classic()
#     )
#     return fig

# plot

In [164]:
# # df = df_plot.head(100)
# df = df_plot
# fig = (
#     p9.ggplot()
#     + p9.geom_bar(
#         data=df,
#         mapping=p9.aes(x="mut_info", y="mut_ratio_mean", fill="treatment"),
#         stat=p9.stats.stat_identity,
#         position=p9.positions.position_dodge(width=0.9, preserve="total"),  # Bar width
#         color="black",
#         size=0.1,
#         raster=False,
#     )
#     + p9.geom_point(
#         data=df,
#         mapping=p9.aes(
#             x="mut_info",
#             y="mut_ratio",
#             # fill="treatment",
#             group="treatment",
#         ),
#         # color="black",
#         position=p9.positions.position_dodge(width=0.9, preserve="total"),  # Bar width
#         size=0.2,
#     )
#     + p9.scale_x_discrete(name="Off-target sites")
#     + p9.scale_y_continuous(
#         name="Editing ratio by Targeted deep sequencing (%)",
#         # breaks=np.arange(0, df.mut_ratio.max(), round(df.mut_ratio.max() / 10, 1)),
#         labels=lambda L: ["%.1f" % v for v in L],
#     )
#     + p9.coord_flip()
#     + p9.facet_grid("region_id~.", scales="free_y")
#     + p9.theme_classic()
#     # + p9.scale_fill_brewer(type="qualitative", palette="Set2")  # 画share old的时候注释掉这一行不然颜色不够用
#     + p9.ggtitle("Targeted deep sequencing ratio")
# )
# fig

In [165]:
# p9.ggsave(
#     fig,
#     "2022-05-24_HEK4_barplot.pdf",
#     height=df_plot.shape[0] * 0.05,
#     width=4,
#     limitsize=False,
# )

# 数据分布

In [166]:
# df_plot

In [167]:
# print("\nmut_ratio of bases%")
# print(
#     df_plot.query("(treatment != 'ABE') | (treatment == 'ABE' & mut_ratio > 0)")
#     .groupby("treatment")
#     .mut_ratio.describe()[["count", "mean", "min", "max", "25%", "50%", "75%"]]
# )

# print("\nmut_ratio of regions%")
# print(
#     df_plot.query("(treatment != 'ABE') | (treatment == 'ABE' & mut_ratio > 0)")
#     .groupby(["treatment", "region_id"])
#     .mut_ratio.max()
#     .reset_index()
#     .groupby("treatment")
#     .mut_ratio.describe()[["count", "mean", "min", "max", "25%", "50%", "75%"]]
# )
# # .describe()[['count', 'mean', 'min', 'max', '25%', '50%', '75%']]

In [168]:
# # 设置画布尺寸
# fig, ax = plt.subplots(2, 2, figsize=[10, 10])

# # 划分绘图区域
# sns.distplot(df_plot.query("treatment == 'ABE'")["mut_ratio"], ax=ax[0, 0])
# ax[0, 0].set_xlabel("ABE, mut_ratio")


# sns.distplot(df_plot.query("treatment == 'ABE8e'")["mut_ratio"], ax=ax[0, 1])
# ax[0, 1].set_xlabel("ABE8e, mut_ratio")
# # plt.xticks(index,labels)
# # plt.plot(ratio,'b-')

# sns.distplot(df_plot.query("treatment == 'untreat'")["mut_ratio"], ax=ax[1, 0])
# ax[1, 0].set_xlabel("untreat, mut_ratio")

# sns.distplot(df_plot.query("treatment == 'vector'")["mut_ratio"], ax=ax[1, 1])
# ax[1, 1].set_xlabel("vector, mut_ratio")


# fig.suptitle("Target-seq mutation ratio density", fontsize=25, color="b")


# plt.savefig("2022-05-28_density.pdf")
# plt.show()